# MCPサーバーをエージェントから使う

MCP（Model Context Protocol）に対応したサーバーのツールを、part1_2・part1_3で実装したエージェントループから呼び出します。

ここではLangChainを使わず、MCPの公式Python SDK（`mcp`）のクライアントと、Chat Completions APIのFunction callingを直接組み合わせます。MCPサーバーが公開するツールの一覧を取得し、それをそのまま `tools` としてLLMに渡すのがポイントです。

> MCPのクライアントは非同期（`async`）のAPIなので、エージェントループも `async` 版になります。Jupyter ではセルの先頭で `await` や `async with` がそのまま使えます。

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

## MCPサーバーに接続してツール一覧を取得する

まずは、GitHubリポジトリについて質問できる [DeepWiki](https://docs.devin.ai/work-with-devin/deepwiki-mcp) のMCPサーバー（リモート、Streamable HTTP）に接続して、どんなツールがあるかを見てみます。

複数のサーバーをまとめて扱えるように、接続をひとつの `async with` にまとめる関数を用意します。MCPサーバーは `command` + `args`（ローカルのプロセスを起動して標準入出力でつなぐ）か、`url`（リモート）で指定します。

In [ ]:
from contextlib import AsyncExitStack, asynccontextmanager

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.streamable_http import streamablehttp_client


@asynccontextmanager
async def connect_mcp_servers(servers: dict):
    """複数のMCPサーバーに接続し、{ツール名: (セッション, ツール定義)} を返す"""
    async with AsyncExitStack() as stack:
        sessions = {}
        for name, params in servers.items():
            if "url" in params:
                read, write, _ = await stack.enter_async_context(
                    streamablehttp_client(params["url"])
                )
            else:
                read, write = await stack.enter_async_context(
                    stdio_client(
                        StdioServerParameters(command=params["command"], args=params["args"])
                    )
                )
            session = await stack.enter_async_context(ClientSession(read, write))
            await session.initialize()

            tools_result = await session.list_tools()
            for mcp_tool in tools_result.tools:
                sessions[mcp_tool.name] = (session, mcp_tool)
            print(f"[connected] {name}: {len(tools_result.tools)} tools")
        yield sessions

In [ ]:
deepwiki_server = {
    "deepwiki": {"url": "https://mcp.deepwiki.com/mcp"},
}

async with connect_mcp_servers(deepwiki_server) as sessions:
    for name, (_, mcp_tool) in sessions.items():
        print(f"- {name}: {(mcp_tool.description or '').splitlines()[0][:80]}")
        print(f"    parameters: {list(mcp_tool.inputSchema.get('properties', {}).keys())}")

## MCPのツールをChat Completions APIの `tools` に変換する

MCPのツール定義（名前・説明・入力のJSON Schema）は、Chat Completions APIのFunction callingの `tools` とほぼ同じ形です。

In [ ]:
def to_openai_tools(sessions: dict) -> list:
    """MCPのツール定義をChat Completions APIのtools形式に変換する"""
    return [
        {
            "type": "function",
            "function": {
                "name": mcp_tool.name,
                "description": mcp_tool.description or "",
                "parameters": mcp_tool.inputSchema,
            },
        }
        for _, mcp_tool in sessions.values()
    ]

## エージェントループ（async版）

part1_2・part1_3の `agent_loop` とほぼ同じです。違いは、ツールの実行がMCPサーバーへの `await session.call_tool(...)` になることだけです。

In [ ]:
import json

from openai import OpenAI

client = OpenAI()


async def agent_loop(messages: list, sessions: dict, max_iterations: int = 20) -> str | None:
    """LLMがツールを使いたいと応答する限り、MCPサーバーのツールを実行して結果を渡し続ける"""
    tools = to_openai_tools(sessions)

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model="gpt-5.6-luna",
            messages=messages,
            tools=tools,
            reasoning_effort="none",
        )
        response_message = response.choices[0].message
        messages.append(response_message.to_dict())

        # ツールを使わない応答なら、それが最終的な回答
        if not response_message.tool_calls:
            return response_message.content

        # ツールを使いたいという応答なら、MCPサーバーのツールを呼び出して結果をmessagesに追加する
        for tool_call in response_message.tool_calls:
            tool_name = tool_call.function.name
            tool_args = json.loads(tool_call.function.arguments)
            print(f"[tool] {tool_name}({json.dumps(tool_args, ensure_ascii=False)[:100]})")

            session, _ = sessions[tool_name]
            result = await session.call_tool(tool_name, tool_args)
            result_text = "\n".join(
                content.text for content in result.content if content.type == "text"
            )
            print(f"  -> {result_text[:200]!r}")

            messages.append(
                {
                    "tool_call_id": tool_call.id,
                    "role": "tool",
                    "name": tool_name,
                    "content": result_text,
                }
            )

    return "（反復回数の上限に達しました）"

## DeepWikiに質問する

In [ ]:
messages = [
    {
        "role": "developer",
        "content": "GitHubで管理されているOSSについて質問された場合は、ask_questionツールを使用して回答してください。",
    },
    {"role": "user", "content": "OSSのPlaywrightの使い方を教えて"},
]

async with connect_mcp_servers(deepwiki_server) as sessions:
    answer = await agent_loop(messages, sessions)

print(answer)

## Serenaを追加する（ローカルのソースコードを検索）

[Serena](https://github.com/oraios/serena) は、ファイルの検索・編集などプログラミングに必要なツールがそろったMCPサーバーです。ローカルのプロセスとして起動し、標準入出力で接続します（`uvx` で取得するため、初回は起動に時間がかかります）。

2つのサーバーのツールを1つのエージェントに渡すと、質問の内容に応じてLLMがサーバーを選びます。

In [ ]:
from pathlib import Path

project_dir = str(Path("..").resolve())

servers = {
    "serena": {
        "command": "uvx",
        "args": [
            "--from",
            "git+https://github.com/oraios/serena",
            "serena",
            "start-mcp-server",
            "--project",
            project_dir,
        ],
    },
    "deepwiki": {"url": "https://mcp.deepwiki.com/mcp"},
}

messages = [
    {
        "role": "developer",
        "content": "GitHubで管理されているOSSについて質問された場合は、ask_questionツールを使用して回答してください。",
    },
    {"role": "user", "content": "ローカルでcreate_agent関数を使っているソースは"},
]

async with connect_mcp_servers(servers) as sessions:
    answer = await agent_loop(messages, sessions)

print(answer)

## 自作のMCPサーバーを使う

MCPサーバーはPythonで自作できます。`app/random_number_mcp.py` は、ランダムな数字を生成するツールを1つ持つMCPサーバーです。`uv run python -m app.random_number_mcp` で起動すると、標準入出力でMCPのやり取りをします。

In [ ]:
random_number_server = {
    "random-number": {
        "command": "uv",
        "args": ["--directory", project_dir, "run", "python", "-m", "app.random_number_mcp"],
    },
}

messages = [
    {"role": "user", "content": "ランダムな数字を3つ言って"},
]

async with connect_mcp_servers(random_number_server) as sessions:
    answer = await agent_loop(messages, sessions)

print(answer)

## まとめ

- MCPサーバーは「ツールの一覧」と「ツールの呼び出し」を標準化したもので、エージェント側から見ると Function calling のツールがサーバーから提供されているだけです
- そのため、エージェントループの実装はほとんど変わらず、ツールの実行がMCPクライアント経由になるだけです
- LangChainでは `langchain-mcp-adapters` の `MultiServerMCPClient` を使うと、MCPのツールをそのまま `create_agent` に渡せます（Streamlitでの例: `pages/partX_4_mcp.py`、`pages/partX_5_custom_mcp.py`）